component 단독 OOF

cw_none : 0.740042
cw_1_5 : 0.740141
cw_2_0 : 0.740196
cw_2_5 : 0.740205

단독으로는 나쁘지 않은데, 챔피언이랑 섞었을 때:

best_blend_oof = 0.740771
best_w_cw = 0.0
improvement = 0.0

즉 모든 class_weight 변형 모델에 대해:

챔피언 4-seed에 섞을 이유 없음


현재 챔피언:

Champion 4-seed OOF ≈ 0.7407705075
LB ≈ 0.74179

class_weight 모델들은 단독 OOF가 0.7402 근처라 꽤 준수하지만, 이미 챔피언 안에 들어있는 CatBoost component들과 예측 순위가 너무 비슷함 새로운 diversity를 못 줌.

판단

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from category_encoders import TargetEncoder

In [2]:
# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

SAVE_DIR = OOF_DIR / "combo_te_v1_cat_weight_sweep_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "cat_weight_sweep"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)
print("SUB_SAVE_DIR:", SUB_SAVE_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_cat_weight_sweep_seed42
SUB_SAVE_DIR: /mnt/c/dev/my_ml_project/submissions/cat_weight_sweep


In [3]:
# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

SAVE_DIR = OOF_DIR / "combo_te_v1_cat_weight_sweep_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "cat_weight_sweep"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)
print("SUB_SAVE_DIR:", SUB_SAVE_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_cat_weight_sweep_seed42
SUB_SAVE_DIR: /mnt/c/dev/my_ml_project/submissions/cat_weight_sweep


In [4]:
def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_main_cat_params(class_weight_value, seed=42):
    """
    class_weight_value:
    - None: class_weights 사용 안 함
    - float: [1, class_weight_value]
    """
    params = dict(
        iterations=2000,
        learning_rate=0.02498214961001344,
        depth=8,
        l2_leaf_reg=18.591182129683194,
        random_strength=0.32969640414889206,
        bagging_temperature=4.535604806522509,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=seed,
        verbose=100,
        allow_writing_files=False,
    )

    if class_weight_value is not None:
        params["class_weights"] = [1, class_weight_value]

    return params

In [6]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [7]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

# 기존 champion data_preprocessing 함수 사용
X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

# combo 문자열 원본 제거
X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

# 컬럼 정렬
X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)
    X_test_stack[col] = X_test_stack[col].astype(str)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)
print("cat_cols:", len(cat_cols))

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)
cat_cols: 54


In [8]:
def train_cat_weight_oof_test(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    class_weight_name,
    class_weight_value,
    n_splits=5,
    fold_seed=42,
    save_dir=None,
):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_seed
    )

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    fold_scores = []
    best_iterations = []

    print("\n" + "=" * 80)
    print(f"Class weight experiment: {class_weight_name}")
    print("class_weight_value:", class_weight_value)
    print("=" * 80)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ {class_weight_name} Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(
            **make_main_cat_params(
                class_weight_value=class_weight_value,
                seed=42
            )
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=100,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_auc)
        best_iterations.append(model.best_iteration_)

        print(f"{class_weight_name} Fold {fold} AUC:", fold_auc)
        print("best_iteration:", model.best_iteration_)

        test_pred += model.predict_proba(X_test_stack)[:, 1] / n_splits

        if save_dir is not None:
            np.save(save_dir / f"{class_weight_name}_partial_oof_fold{fold}.npy", oof)
            np.save(save_dir / f"{class_weight_name}_partial_test_fold{fold}.npy", test_pred)

    oof_auc = roc_auc_score(y_stack, oof)

    print("\n================ RESULT ================")
    print("class_weight_name:", class_weight_name)
    print("fold_scores:", fold_scores)
    print("mean_fold_auc:", np.mean(fold_scores))
    print("oof_auc:", oof_auc)
    print("best_iterations:", best_iterations)

    if save_dir is not None:
        np.save(save_dir / f"{class_weight_name}_oof.npy", oof)
        np.save(save_dir / f"{class_weight_name}_test_pred.npy", test_pred)

        pd.DataFrame([{
            "class_weight_name": class_weight_name,
            "class_weight_value": class_weight_value,
            "oof_auc": oof_auc,
            "mean_fold_auc": np.mean(fold_scores),
            "fold_scores": str(fold_scores),
            "best_iterations": str(best_iterations),
        }]).to_csv(
            save_dir / f"{class_weight_name}_summary.csv",
            index=False
        )

    return oof, test_pred, fold_scores

In [9]:
class_weight_experiments = [
    ("cw_none", None),
    ("cw_1_5", 1.5),
    ("cw_2_0", 2.0),
    ("cw_2_5", 2.5),
    # 현재값 확인용. 시간이 아까우면 생략 가능.
    # ("cw_2_87", POS_WEIGHT),
]

In [10]:
cw_results = {}

for name, value in class_weight_experiments:
    oof, test_pred, fold_scores = train_cat_weight_oof_test(
        X_stack=X_stack,
        X_test_stack=X_test_stack,
        y_stack=y_stack,
        cat_cols=cat_cols,
        class_weight_name=name,
        class_weight_value=value,
        n_splits=N_SPLITS,
        fold_seed=FOLD_SEED,
        save_dir=SAVE_DIR,
    )

    cw_results[name] = {
        "oof": oof,
        "test_pred": test_pred,
        "oof_auc": roc_auc_score(y_stack, oof),
        "fold_scores": fold_scores,
        "class_weight_value": value,
    }

    print("\nSaved:", name)
    print("OOF:", cw_results[name]["oof_auc"])


Class weight experiment: cw_none
class_weight_value: None

================ cw_none Fold 1 ================
0:	test: 0.7269587	best: 0.7269587 (0)	total: 324ms	remaining: 10m 47s
100:	test: 0.7357103	best: 0.7357103 (100)	total: 29.8s	remaining: 9m 19s
200:	test: 0.7374764	best: 0.7374764 (200)	total: 51s	remaining: 7m 36s
300:	test: 0.7379795	best: 0.7379795 (300)	total: 1m 4s	remaining: 6m 5s
400:	test: 0.7381425	best: 0.7381551 (396)	total: 1m 17s	remaining: 5m 10s
500:	test: 0.7382430	best: 0.7382461 (486)	total: 1m 31s	remaining: 4m 32s
600:	test: 0.7382522	best: 0.7382555 (584)	total: 1m 44s	remaining: 4m 3s
700:	test: 0.7382490	best: 0.7382650 (661)	total: 1m 57s	remaining: 3m 38s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.738264951
bestIteration = 661

Shrink model to first 662 iterations.
cw_none Fold 1 AUC: 0.738264951010976
best_iteration: 661

================ cw_none Fold 2 ================
0:	test: 0.7248865	best: 0.7248865 (0)	total: 131ms	rema

In [11]:
cw_summary = pd.DataFrame([
    {
        "name": name,
        "class_weight_value": result["class_weight_value"],
        "oof_auc": result["oof_auc"],
        "fold_scores": result["fold_scores"],
    }
    for name, result in cw_results.items()
]).sort_values("oof_auc", ascending=False)

display(cw_summary)

cw_summary.to_csv(SAVE_DIR / "class_weight_sweep_summary.csv", index=False)

,name,class_weight_value,oof_auc,fold_scores
3,cw_2_5,2.5,0.740205,"[0.7380597360442653, 0.7430323357728185, 0.740..."
2,cw_2_0,2.0,0.740196,"[0.7382936270044134, 0.7430127672755079, 0.740..."
1,cw_1_5,1.5,0.740141,"[0.7382733442509979, 0.7429844187167997, 0.740..."
0,cw_none,NaN,0.740042,"[0.738264951010976, 0.7428286143233438, 0.7401..."


In [12]:
seed42_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_4seed_oof = (
    seed42_oof +
    seed2024_oof +
    seed777_oof +
    seed999_oof
) / 4

champion_4seed_score = roc_auc_score(y_stack, champion_4seed_oof)

print("Champion 4-seed OOF:", champion_4seed_score)

Champion 4-seed OOF: 0.7407705074934163


In [13]:
blend_rows = []

champ_rank_oof = rank01(champion_4seed_oof)

for name, result in cw_results.items():
    cw_oof = result["oof"]
    cw_rank_oof = rank01(cw_oof)

    best_score = champion_4seed_score
    best_w = 0.0
    best_blend = champion_4seed_oof.copy()

    for w_cw in np.arange(0.00, 0.31, 0.01):
        blend = (
            (1 - w_cw) * champ_rank_oof +
            w_cw * cw_rank_oof
        )

        score = roc_auc_score(y_stack, blend)

        if score > best_score:
            best_score = score
            best_w = w_cw
            best_blend = blend.copy()

    blend_rows.append({
        "name": name,
        "class_weight_value": result["class_weight_value"],
        "cw_oof": result["oof_auc"],
        "best_blend_oof": best_score,
        "best_w_cw": best_w,
        "improvement": best_score - champion_4seed_score,
    })

    np.save(
        SAVE_DIR / f"{name}_best_blend_oof.npy",
        best_blend
    )

blend_summary = pd.DataFrame(blend_rows).sort_values(
    "best_blend_oof",
    ascending=False
)

display(blend_summary)

blend_summary.to_csv(
    SAVE_DIR / "champion4seed_class_weight_blend_summary.csv",
    index=False
)

print("Champion 4-seed OOF:", champion_4seed_score)

,name,class_weight_value,cw_oof,best_blend_oof,best_w_cw,improvement
0,cw_none,NaN,0.740042,0.740771,0.0,0.0
1,cw_1_5,1.5,0.740141,0.740771,0.0,0.0
2,cw_2_0,2.0,0.740196,0.740771,0.0,0.0
3,cw_2_5,2.5,0.740205,0.740771,0.0,0.0


Champion 4-seed OOF: 0.7407705074934163
